In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_br.order")

In [0]:
from pyspark.sql.functions import col, to_date
order_df = order_df.select(col("ord").alias("orderno"),
                               col("odate").alias("orderdate"),
                               col("wt").alias("orderweight"),
                               col("ordervalue"),
                               col("productcategory"),
                               col("productname"),
                               col("productsize"),
                               col("productvalue"),
                               col("productweight"),
                               col("noofitems"),
                               col("sendername"),
                               col("sendercity"),
                               col("senderstate"),
                               col("receivername"),
                               col("receivercity"),
                               col("receiverstate"),
                               col("estcost").alias("estimatedcost"),
                               col("shipno").alias("shipmentno"),
                               col("dcost").alias("deliverycost"),
                               col("carriername"),
                               col("carriercity"),
                               col("carrierstate"))

                               

In [0]:

order_df = order_df.withColumn("ingestion_time",current_timestamp())
order_df = convert_to_timestamp(order_df, "orderdate")

In [0]:
target_columns = ["orderweight", "ordervalue","productvalue","productweight","estimatedcost","deliverycost"]

# Apply the UDF in a loop
for col_name in target_columns:
     order_df = convert_to_float(order_df, col_name)

In [0]:
target_columns = ["productsize", "noofitems"]

# Apply the UDF in a loop
for col_name in target_columns:
     order_df = convert_to_int(order_df, col_name)

In [0]:
order_df.createOrReplaceTempView("temp_order")

In [0]:
%sql
MERGE INTO dev.tracking_sl.order tgt 
USING ( 
  SELECT DISTINCT * 
  FROM temp_order 
) src 
ON tgt.orderno = src.orderno 
WHEN NOT MATCHED THEN 
  INSERT *